[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hien078/applied-mathematics-foundation/blob/master/numerical_computing/03_conditioning_and_condition_numbers/first_principles.ipynb)

# Topic 03: Conditioning and Condition Numbers

## 1. First-Principles Intuition & Motivation

Suppose you must report the answer to a question whose input data you know only to 16 digits — which is the *best case* for binary64, because the very act of storing $x$ commits a relative error of size $u$. Even a hypothetical oracle that computed $f$ in exact arithmetic would return $f(\mathrm{fl}(x))$, not $f(x)$. How far apart are those two numbers?

That question has nothing to do with algorithms, programming languages, or hardware. It is a property of the *map* $f$ and the *point* $x$. Its answer is the **condition number**, and it sets a hard floor on the accuracy of every possible implementation.

Topic 02 asked "how much error does my algorithm commit?" (backward error). This module asks "how much does the problem amplify any error at all?" (conditioning). The two multiply:

$$
\text{forward error} \; \lesssim \; \underbrace{\kappa}_{\text{problem's fault}} \times \underbrace{\eta}_{\text{algorithm's fault}}
$$

Everything in numerical linear algebra — why we use QR instead of normal equations, why `solve` beats `inv`, why ridge regression stabilizes regression, why deep-network Hessians make plain gradient descent crawl — is an application of this single inequality.

### A physical analogy: the lever

A condition number is a lever arm. Push the input end by a relative amount $\varepsilon$; the output end moves by $\kappa \varepsilon$. A well-conditioned problem ($\kappa \approx 1$) is a rigid rod: you get out what you put in. An ill-conditioned problem ($\kappa = 10^{12}$) is a lever with a $10^{12}$-to-one mechanical advantage — a nanometre of input jitter becomes a kilometre of output swing.

The crucial consequence: **you cannot fix a long lever by pushing more carefully.** In binary64 you cannot push more carefully than $u \approx 1.1 \times 10^{-16}$, because that is the resolution of the input representation itself. If $\kappa = 10^{12}$, the *best conceivable* answer carries relative error $10^{-4}$ — four correct digits, no matter what algorithm, what library, what compiler flags.

The engineering responses are therefore never "compute harder". They are: (i) reformulate the problem so the map itself is better conditioned (QR instead of normal equations, orthogonal bases instead of monomials), (ii) change the problem you are solving (regularization), or (iii) increase the input precision (extended precision, better measurements).

## 2. Rigorous Mathematical Definitions & Theorem Statements

### Definition 2.1 (Absolute and relative condition number)

Let $f : \mathbb{R}^{n} \to \mathbb{R}^{m}$ and let $\delta x$ denote a perturbation of the input $x$.

**Absolute condition number:**

$$
\hat{\kappa}_f(x) = \lim_{\varepsilon \to 0^{+}} \; \sup_{\Vert \delta x \Vert \le \varepsilon} \frac{\Vert f(x + \delta x) - f(x) \Vert}{\Vert \delta x \Vert}
$$

**Relative condition number:**

$$
\kappa_f(x) = \lim_{\varepsilon \to 0^{+}} \; \sup_{\Vert \delta x \Vert \le \varepsilon} \left( \frac{\Vert f(x + \delta x) - f(x) \Vert}{\Vert f(x) \Vert} \Big/ \frac{\Vert \delta x \Vert}{\Vert x \Vert} \right)
$$

Because floating-point error is *relative* by construction (Topic 01's axiom), $\kappa_f$ — not $\hat{\kappa}_f$ — is the quantity that governs achievable digits. A problem is called **well-conditioned** when $\kappa_f$ is modest ($1$ to $10^{3}$, say) and **ill-conditioned** when it is large.

### Theorem 2.2 (Derivative formula for the condition number)

If $f$ is differentiable at $x$ with Jacobian $J_f(x)$, then

$$
\hat{\kappa}_f(x) = \Vert J_f(x) \Vert, \qquad \kappa_f(x) = \frac{\Vert J_f(x) \Vert \, \Vert x \Vert}{\Vert f(x) \Vert}
$$

In the scalar case $f : \mathbb{R} \to \mathbb{R}$ this collapses to the formula every practitioner should memorize:

$$
\kappa_f(x) = \left\vert \frac{x f'(x)}{f(x)} \right\vert
$$

**Worked values.**

| $f(x)$ | $\kappa_f(x)$ | Ill-conditioned where |
|---|---|---|
| $x^{a}$ | $\vert a \vert$ | never (for moderate $a$) |
| $\sqrt{x}$ | $1/2$ | never — square root *halves* error |
| $e^{x}$ | $\vert x \vert$ | large $\vert x \vert$ |
| $\log x$ | $1 / \vert \log x \vert$ | $x \to 1$ |
| $\sin x$ | $\vert x \cot x \vert$ | near zeros of $\sin$, i.e. $x \to k\pi$ |
| $x_1 - x_2$ | $\frac{\vert x_1 \vert + \vert x_2 \vert}{\vert x_1 - x_2 \vert}$ | $x_1 \approx x_2$ (cancellation) |

The last row is Topic 02's cancellation bound, now identified for what it is: subtraction is an *ill-conditioned map* near the diagonal. Nothing was ever wrong with the hardware.

### Definition 2.3 (Matrix condition number)

For a nonsingular $A \in \mathbb{R}^{n \times n}$ and any submultiplicative matrix norm,

$$
\kappa(A) = \Vert A \Vert \, \Vert A^{-1} \Vert
$$

Basic properties, all immediate from the definition:

1. $\kappa(A) \ge \Vert A A^{-1} \Vert = \Vert I \Vert \ge 1$ — conditioning is never better than perfect.
2. $\kappa(cA) = \kappa(A)$ for any $c \ne 0$ — it is a *ratio* of scales, hence scale invariant. (Contrast $\det(cA) = c^{n}\det A$.)
3. In the 2-norm, with singular values $\sigma_1 \ge \cdots \ge \sigma_n \gt 0$,

$$
\kappa_2(A) = \frac{\sigma_{\max}(A)}{\sigma_{\min}(A)}
$$

4. For orthogonal $Q$: $\kappa_2(Q) = 1$ — orthogonal transformations neither amplify nor damp relative error. This one fact is why virtually every stable dense algorithm (QR, Householder, Givens, SVD) is built out of orthogonal operations.
5. For symmetric $A$: $\kappa_2(A) = \frac{\max_i \vert \lambda_i \vert}{\min_i \vert \lambda_i \vert}$.
6. If $A$ is singular, set $\kappa(A) = \infty$.

### Theorem 2.4 (Perturbation bound for linear systems)

Let $Ax = b$ with $A$ nonsingular, and let $(A + \Delta A)\hat{x} = b + \Delta b$. If $\Vert A^{-1} \Vert \Vert \Delta A \Vert \lt 1$ then $A + \Delta A$ is nonsingular and

$$
\frac{\Vert \hat{x} - x \Vert}{\Vert x \Vert} \le \frac{\kappa(A)}{1 - \kappa(A) \frac{\Vert \Delta A \Vert}{\Vert A \Vert}} \left( \frac{\Vert \Delta A \Vert}{\Vert A \Vert} + \frac{\Vert \Delta b \Vert}{\Vert b \Vert} \right)
$$

To first order, dropping the denominator,

$$
\frac{\Vert \hat{x} - x \Vert}{\Vert x \Vert} \; \lesssim \; \kappa(A) \left( \frac{\Vert \Delta A \Vert}{\Vert A \Vert} + \frac{\Vert \Delta b \Vert}{\Vert b \Vert} \right)
$$

**Corollary (residual does not certify accuracy).** With $r = b - A\hat{x}$,

$$
\frac{\Vert \hat{x} - x \Vert}{\Vert x \Vert} \le \kappa(A) \, \frac{\Vert r \Vert}{\Vert b \Vert}
$$

and this bound is attained. A residual at the level $10^{-16}$ therefore guarantees only $10^{-16}\kappa(A)$ forward accuracy — for $\kappa = 10^{10}$, six digits.

**Theorem 2.4b (Kahan / Gastinel — distance to singularity).**

$$
\frac{1}{\kappa_2(A)} = \min \left\{ \frac{\Vert \Delta A \Vert_2}{\Vert A \Vert_2} \; : \; A + \Delta A \text{ is singular} \right\} = \frac{\sigma_{\min}(A)}{\Vert A \Vert_2}
$$

The reciprocal condition number is *exactly* the relative distance to the nearest singular matrix. Ill-conditioning means "nearly unsolvable", in the most literal metric sense.

### Theorem 2.5 (Least squares: conditioning and the normal-equation trap)

For $X \in \mathbb{R}^{m \times n}$ of full column rank $n \le m$, define $\kappa_2(X) = \sigma_{\max}(X)/\sigma_{\min}(X)$ (via the nonzero singular values) and let $\beta$ minimize $\Vert X\beta - y \Vert_2$ with residual $r = y - X\beta$ and angle $\theta$ defined by $\sin\theta = \Vert r \Vert_2 / \Vert y \Vert_2$. Then for a perturbation of $X$ of relative size $\epsilon$,

$$
\frac{\Vert \Delta\beta \Vert_2}{\Vert \beta \Vert_2} \; \lesssim \; \left( 2\kappa_2(X) + \kappa_2(X)^{2} \tan\theta \right) \epsilon
$$

Two regimes: for a **consistent / nearly consistent** system ($\tan\theta \approx 0$) least squares behaves like $\kappa_2(X)$; for a **large-residual** fit the sensitivity genuinely reaches $\kappa_2(X)^{2}$ — that part is intrinsic, not an artifact of the method.

**The normal-equation trap.** The Gram matrix satisfies

$$
\kappa_2(X^{\top}X) = \kappa_2(X)^{2}
$$

*always*. Solving $X^{\top}X\beta = X^{\top}y$ therefore hands a $\kappa^{2}$-conditioned system to the linear solver even when the underlying least-squares problem is only $\kappa$-conditioned. QR and SVD-based solvers (`numpy.linalg.lstsq`, `scipy.linalg.lstsq`) never form $X^{\top}X$ and so operate at the intrinsic level.

### Theorem 2.6 (Conditioning of the eigenvalue problem)

Let $\lambda$ be a **simple** eigenvalue of $A$ with unit right eigenvector $x$ ($Ax = \lambda x$) and unit left eigenvector $y$ ($y^{*}A = \lambda y^{*}$). For a perturbation $A + \varepsilon E$ with $\Vert E \Vert_2 = 1$, the perturbed eigenvalue satisfies

$$
\lambda(\varepsilon) = \lambda + \varepsilon \, \frac{y^{*} E x}{y^{*} x} + O(\varepsilon^{2}), \qquad \text{so} \qquad \hat{\kappa}_{\lambda} = \frac{1}{\vert y^{*} x \vert} =: \sec\vartheta
$$

where $\vartheta$ is the angle between the left and right eigenvectors. The eigenvalue condition number is the **secant of the angle between left and right eigenvectors** — not $\kappa(A)$.

**Consequences.**

- **Normal matrices** ($A^{*}A = AA^{*}$, includes symmetric and orthogonal) have $y = x$, hence $\hat{\kappa}_{\lambda} = 1$: eigenvalues of symmetric matrices are perfectly conditioned, and every eigenvalue moves by at most $\Vert \Delta A \Vert_2$ (Weyl's inequality).
- **Bauer–Fike.** If $A = V\Lambda V^{-1}$ is diagonalizable, every eigenvalue $\mu$ of $A + \Delta A$ satisfies $\min_i \vert \mu - \lambda_i \vert \le \kappa_2(V) \, \Vert \Delta A \Vert_2$. It is the conditioning of the *eigenvector matrix*, not of $A$, that governs.
- **Defective/near-defective matrices** are catastrophically ill-conditioned: a Jordan block of size $k$ perturbed by $\varepsilon$ moves its eigenvalues by $\varepsilon^{1/k}$ — a $10^{-16}$ perturbation of a $4 \times 4$ Jordan block moves eigenvalues by $10^{-4}$.
- **Eigenvectors** have their own condition number, governed by the eigenvalue *gap*: $\hat{\kappa}_{x_i} \approx 1 / \min_{j \ne i} \vert \lambda_i - \lambda_j \vert$. Clustered eigenvalues make individual eigenvectors meaningless while their invariant subspace stays perfectly well determined.

### Definition 2.7 (Stability, and the master identity)

- An algorithm $\tilde{f}$ is **backward stable** if for every $x$ there is $\Delta x$ with $\tilde{f}(x) = f(x + \Delta x)$ and $\frac{\Vert \Delta x \Vert}{\Vert x \Vert} = O(u)$.
- It is **stable (mixed forward–backward)** if $\tilde{f}(x) = f(x + \Delta x)(1 + O(u))$ with $\frac{\Vert \Delta x \Vert}{\Vert x \Vert} = O(u)$ — "nearly the right answer to nearly the right question".

**Master identity.** For a backward stable algorithm applied to a problem of relative condition number $\kappa_f$,

$$
\frac{\Vert \tilde{f}(x) - f(x) \Vert}{\Vert f(x) \Vert} \; = \; O\!\left( \kappa_f(x) \, u \right)
$$

**Wilkinson's rule of thumb.** In binary64,

$$
\text{correct decimal digits} \; \approx \; 16 - \log_{10} \kappa_f(x)
$$

so $\kappa = 10^{8}$ costs half your digits and $\kappa \ge 10^{16}$ leaves none. This is the single most useful sentence in the module: *look up $\kappa$, subtract its logarithm from your precision, and you know your answer before you compute it.*

## 3. Step-by-Step Mathematical Proofs & Derivations

### Derivation 3.1: The scalar condition number from a first-order Taylor expansion

**Claim.** For differentiable $f$ with $f(x) \ne 0$, $x \ne 0$: $\kappa_f(x) = \vert x f'(x) / f(x) \vert$.

**Proof.** Perturb $x \mapsto x(1 + \varepsilon)$, i.e. $\delta x = \varepsilon x$ with relative size $\vert \varepsilon \vert$. Taylor:

$$
f(x + \varepsilon x) = f(x) + \varepsilon x f'(x) + O(\varepsilon^{2})
$$

Hence the relative output change is

$$
\frac{f(x + \varepsilon x) - f(x)}{f(x)} = \varepsilon \, \frac{x f'(x)}{f(x)} + O(\varepsilon^{2})
$$

Dividing by the relative input change $\varepsilon$ and taking $\varepsilon \to 0$ gives the ratio $\vert x f'(x)/f(x) \vert$, and the supremum over the two signs of $\varepsilon$ is attained. $\blacksquare$

**Application — roots of a polynomial.** Let $p(z) = \prod_{i}(z - r_i)$ and view a root $r$ as a function of a coefficient $a_k$. Implicit differentiation of $p(r) = 0$ gives $\frac{\partial r}{\partial a_k} = -\frac{r^{k}}{p'(r)}$, so

$$
\kappa_{r, a_k} = \left\vert \frac{a_k r^{k-1}}{p'(r)} \right\vert
$$

For a *multiple* root $p'(r) = 0$ and the condition number is infinite — consistent with the $\varepsilon^{1/k}$ perturbation law. **Wilkinson's polynomial** $p(z) = \prod_{k=1}^{20}(z - k)$ has simple, well-separated roots, yet perturbing the coefficient of $z^{19}$ by $2^{-23}$ (one binary32 ulp) moves the root at $z = 20$ by about $0.9$: the derived condition number is $\approx 5 \times 10^{13}$. Well-separated roots are *not* the same as a well-conditioned root-from-coefficients map.

### Derivation 3.2: Proof of the linear-system perturbation theorem

**Setup.** $Ax = b$, $(A + \Delta A)(x + \delta x) = b + \Delta b$. Expand:

$$
Ax + A\,\delta x + \Delta A\, x + \Delta A\, \delta x = b + \Delta b
$$

Cancel $Ax = b$ and solve for $\delta x$:

$$
(A + \Delta A)\, \delta x = \Delta b - \Delta A\, x \quad \Longrightarrow \quad \delta x = (A + \Delta A)^{-1} \left( \Delta b - \Delta A\, x \right)
$$

**Step 1 — invertibility.** Write $A + \Delta A = A(I + A^{-1}\Delta A)$. If $\Vert A^{-1}\Delta A \Vert \le \Vert A^{-1} \Vert \Vert \Delta A \Vert \lt 1$, the Neumann series $\sum_k (-A^{-1}\Delta A)^{k}$ converges, so $I + A^{-1}\Delta A$ is invertible with

$$
\Vert (I + A^{-1}\Delta A)^{-1} \Vert \le \frac{1}{1 - \Vert A^{-1} \Vert \Vert \Delta A \Vert}
$$

**Step 2 — bound $\delta x$.** Using $(A + \Delta A)^{-1} = (I + A^{-1}\Delta A)^{-1}A^{-1}$,

$$
\Vert \delta x \Vert \le \frac{\Vert A^{-1} \Vert}{1 - \Vert A^{-1} \Vert \Vert \Delta A \Vert} \left( \Vert \Delta b \Vert + \Vert \Delta A \Vert \Vert x \Vert \right)
$$

**Step 3 — relativize.** Divide by $\Vert x \Vert$ and use $\Vert b \Vert = \Vert Ax \Vert \le \Vert A \Vert \Vert x \Vert$, i.e. $\frac{1}{\Vert x \Vert} \le \frac{\Vert A \Vert}{\Vert b \Vert}$:

$$
\frac{\Vert \delta x \Vert}{\Vert x \Vert} \le \frac{\Vert A^{-1} \Vert \Vert A \Vert}{1 - \Vert A^{-1} \Vert \Vert A \Vert \frac{\Vert \Delta A \Vert}{\Vert A \Vert}} \left( \frac{\Vert \Delta b \Vert}{\Vert b \Vert} + \frac{\Vert \Delta A \Vert}{\Vert A \Vert} \right)
$$

which is the stated bound with $\kappa(A) = \Vert A \Vert \Vert A^{-1} \Vert$. $\blacksquare$

**Sharpness.** Choose $\Delta b$ along the *left* singular vector $u_n$ belonging to $\sigma_{\min}$ and $b$ along $u_1$: then $\Vert A^{-1}\Delta b \Vert = \Vert \Delta b \Vert/\sigma_{\min}$ while $\Vert x \Vert = \Vert b \Vert / \sigma_{\max}$, and the ratio is exactly $\kappa_2(A)$. The bound is attained, so $\kappa$ is not a pessimistic artifact of the proof.

### Derivation 3.3: Why $\kappa_2 = \sigma_{\max}/\sigma_{\min}$ — the ellipsoid picture

**Claim.** $\Vert A \Vert_2 = \sigma_{\max}$, $\Vert A^{-1} \Vert_2 = 1/\sigma_{\min}$, hence $\kappa_2(A) = \sigma_{\max}/\sigma_{\min}$.

**Proof.** Let $A = U\Sigma V^{\top}$ be the SVD. Since $U, V$ are orthogonal and the 2-norm is orthogonally invariant,

$$
\Vert A \Vert_2 = \max_{\Vert v \Vert_2 = 1} \Vert U \Sigma V^{\top} v \Vert_2 = \max_{\Vert w \Vert_2 = 1} \Vert \Sigma w \Vert_2 = \max_i \sigma_i = \sigma_{\max}
$$

Applying the same argument to $A^{-1} = V\Sigma^{-1}U^{\top}$, whose singular values are $1/\sigma_i$, gives $\Vert A^{-1} \Vert_2 = 1/\sigma_{\min}$. $\blacksquare$

**Geometric reading.** $A$ maps the unit sphere onto an ellipsoid with semi-axes $\sigma_1 \ge \cdots \ge \sigma_n$ along the columns of $U$. Then:

- $\kappa_2(A)$ is the **aspect ratio** of that ellipsoid.
- Solving $Ax = b$ pulls back from the ellipsoid to the sphere; a displacement of $b$ along the *shortest* axis is stretched by $1/\sigma_{\min}$, while $\Vert x \Vert$ itself is set by the component of $b$ along the *longest* axis. The worst-case ratio of these two effects is precisely $\sigma_{\max}/\sigma_{\min}$.
- Kahan's theorem (2.4b) is now visible: to make $A$ singular you must collapse the shortest axis, which costs a perturbation of size exactly $\sigma_{\min}$; relative to $\Vert A \Vert_2 = \sigma_{\max}$, that is $1/\kappa_2$.

**Canonical catastrophes.**

| Matrix | Entries | $\kappa_2$ growth |
|---|---|---|
| Hilbert $H_n$ | $h_{ij} = 1/(i+j-1)$ | $\approx e^{3.5n}$; $\kappa_2(H_{12}) \approx 1.6 \times 10^{16}$ |
| Vandermonde, real nodes | $v_{ij} = x_i^{\,j-1}$ | exponential in $n$ for clustered nodes |
| Monomial-basis polynomial fit, degree $d$ | design matrix | $\approx 10^{d}$ on $[0, 1]$ |
| Nearly duplicated feature columns | $x_2 = x_1 + \epsilon z$ | $\approx \Vert X \Vert / \epsilon$ |

Every one of them is repaired by changing basis: Legendre/Chebyshev polynomials instead of monomials, orthogonalized or standardized features instead of raw ones.

### Derivation 3.4: The normal equations square the condition number

**Claim.** For $X \in \mathbb{R}^{m \times n}$ of full column rank, $\kappa_2(X^{\top}X) = \kappa_2(X)^{2}$.

**Proof.** Let $X = U\Sigma V^{\top}$ be the thin SVD, $\Sigma = \mathrm{diag}(\sigma_1, \dots, \sigma_n)$ with $U^{\top}U = I_n$. Then

$$
X^{\top}X = V\Sigma U^{\top} U \Sigma V^{\top} = V \Sigma^{2} V^{\top}
$$

which is an eigendecomposition of a symmetric positive definite matrix with eigenvalues $\sigma_i^{2}$. Hence

$$
\kappa_2(X^{\top}X) = \frac{\max_i \sigma_i^{2}}{\min_i \sigma_i^{2}} = \left( \frac{\sigma_{\max}}{\sigma_{\min}} \right)^{2} = \kappa_2(X)^{2} \qquad \blacksquare
$$

**Accuracy consequence.** A backward stable solve of the normal equations delivers forward error $O(\kappa_2(X)^{2} u)$; QR-based least squares delivers $O\!\left( (\kappa_2(X) + \kappa_2(X)^{2}\tan\theta) u \right)$, which for small residuals is $O(\kappa_2(X) u)$. In binary64 with $\kappa_2(X) = 10^{8}$:

$$
\text{normal equations: } 10^{16} \cdot u \approx 1 \; (\text{no digits}), \qquad \text{QR: } 10^{8} \cdot u \approx 10^{-8} \; (\text{8 digits})
$$

**Concrete failure — the Läuchli matrix.** Take

$$
X = \begin{pmatrix} 1 & 1 \\ \epsilon & 0 \\ 0 & \epsilon \end{pmatrix}, \qquad X^{\top}X = \begin{pmatrix} 1 + \epsilon^{2} & 1 \\ 1 & 1 + \epsilon^{2} \end{pmatrix}
$$

For $\epsilon \lt \sqrt{u}$ (about $1.5 \times 10^{-8}$ in binary64), $\mathrm{fl}(1 + \epsilon^{2}) = 1$ and the computed Gram matrix is exactly singular — Cholesky fails outright — while $X$ itself has $\kappa_2(X) \approx \sqrt{2}/\epsilon \approx 10^{8}$, entirely solvable by QR. The information destroyed by forming $X^{\top}X$ was never recoverable afterwards.

```python
# Illustrative only (not executed here):
# import numpy as np
# eps = 1e-8
# X = np.array([[1.0, 1.0], [eps, 0.0], [0.0, eps]])
# np.linalg.cond(X)          # ~1.4e8
# np.linalg.cond(X.T @ X)    # ~1e16 or Inf  -> squared
# np.linalg.lstsq(X, y, rcond=None)   # works at kappa(X)
```

### Derivation 3.5: The master identity and Wilkinson's digit rule

**Claim.** If $\tilde{f}$ is backward stable, its relative forward error is $O(\kappa_f(x) u)$.

**Proof.** Backward stability gives $\tilde{f}(x) = f(\tilde{x})$ with $\tilde{x} = x + \Delta x$, $\frac{\Vert \Delta x \Vert}{\Vert x \Vert} \le c\,u$. Apply the definition of the relative condition number to the perturbation $\Delta x$ (legitimate for $u$ small enough that first-order analysis applies):

$$
\frac{\Vert f(\tilde{x}) - f(x) \Vert}{\Vert f(x) \Vert} \le \left( \kappa_f(x) + o(1) \right) \frac{\Vert \Delta x \Vert}{\Vert x \Vert} \le \kappa_f(x) \, c\,u + o(u)
$$

Substituting $\tilde{f}(x) = f(\tilde{x})$ on the left gives the claim. $\blacksquare$

**Taking logarithms.** With $\kappa = 10^{k}$ and $u = 10^{-p}$ (so $p \approx 15.95$ for binary64, $p \approx 7.2$ for binary32, $p \approx 3.3$ for fp16),

$$
\text{relative forward error} \approx 10^{k - p} \quad \Longrightarrow \quad \text{correct digits} \approx p - \log_{10}\kappa
$$

**Precision budget table** (digits of the answer you can trust):

| $\kappa$ | fp16 ($p \approx 3.3$) | bf16 ($p \approx 2.4$) | fp32 ($p \approx 7.2$) | fp64 ($p \approx 16$) |
|---|---|---|---|---|
| $10^{0}$ | 3.3 | 2.4 | 7.2 | 16 |
| $10^{3}$ | 0.3 | none | 4.2 | 13 |
| $10^{6}$ | none | none | 1.2 | 10 |
| $10^{10}$ | none | none | none | 6 |
| $10^{16}$ | none | none | none | none |

Read backwards, the table is a *precision requirements calculator*: to get 6 correct digits from a problem with $\kappa = 10^{6}$ you need at least $p = 12$, i.e. binary64.

**The three-way diagnosis.** When an answer is wrong, compute both quantities:

| Backward error | $\kappa$ | Diagnosis | Fix |
|---|---|---|---|
| $O(u)$ | small | Algorithm and problem both fine — bug elsewhere | debug the code |
| $O(u)$ | huge | Ill-conditioned problem | reformulate / regularize / more precision |
| $\gg u$ | small | Unstable algorithm | change algorithm (Topic 02) |
| $\gg u$ | huge | Both | fix both |

### Derivation 3.6: Regularization as a conditioning repair

**Ridge / Tikhonov.** Replace $\min_\beta \Vert X\beta - y \Vert_2^{2}$ by

$$
\min_{\beta} \; \Vert X\beta - y \Vert_2^{2} + \lambda \Vert \beta \Vert_2^{2}, \qquad \lambda \gt 0
$$

whose normal equations are $(X^{\top}X + \lambda I)\beta = X^{\top}y$.

**Effect on the spectrum.** With $X = U\Sigma V^{\top}$,

$$
X^{\top}X + \lambda I = V(\Sigma^{2} + \lambda I)V^{\top}
$$

so every eigenvalue $\sigma_i^{2}$ is lifted to $\sigma_i^{2} + \lambda$, and

$$
\kappa_2\!\left( X^{\top}X + \lambda I \right) = \frac{\sigma_{\max}^{2} + \lambda}{\sigma_{\min}^{2} + \lambda} \; \le \; \frac{\sigma_{\max}^{2} + \lambda}{\lambda} \; = \; 1 + \frac{\sigma_{\max}^{2}}{\lambda}
$$

The bound is **independent of $\sigma_{\min}$**: even an exactly rank-deficient $X$ yields a finite condition number. Choosing $\lambda = \sqrt{u}\,\sigma_{\max}^{2}$ caps $\kappa$ at $\approx u^{-1/2} = 10^{8}$, restoring half the digits of binary64.

**Equivalent view via the augmented system.** Ridge regression is ordinary least squares on the stacked problem

$$
\tilde{X} = \begin{pmatrix} X \\ \sqrt{\lambda}\, I \end{pmatrix}, \qquad \tilde{y} = \begin{pmatrix} y \\ 0 \end{pmatrix}, \qquad \kappa_2(\tilde{X}) = \sqrt{ \frac{\sigma_{\max}^{2} + \lambda}{\sigma_{\min}^{2} + \lambda} }
$$

which lets QR solve the regularized problem at the *square root* of the Gram-matrix conditioning — the numerically preferred route, and what `sklearn`'s `solver="lsqr"`/`"svd"` paths do.

**The bias–variance price.** In the SVD basis, ridge shrinks each coefficient by the filter factor

$$
f_i = \frac{\sigma_i^{2}}{\sigma_i^{2} + \lambda} \in (0, 1)
$$

Directions with $\sigma_i^{2} \gg \lambda$ pass through untouched; directions with $\sigma_i^{2} \ll \lambda$ — exactly the ones whose coefficients were pure amplified noise — are suppressed. **Regularization is not a numerical trick that gets something for nothing: it changes the estimand, trading bias for a conditioned, low-variance solution.** Truncated SVD is the hard-threshold version of the same filter ($f_i \in \{0, 1\}$).

### Derivation 3.7: Conditioning of a simple eigenvalue

**Setup.** $Ax = \lambda x$, $y^{*}A = \lambda y^{*}$, $\Vert x \Vert_2 = \Vert y \Vert_2 = 1$, $\lambda$ simple (so $y^{*}x \ne 0$). Perturb $A(\varepsilon) = A + \varepsilon E$. Analytic perturbation theory for a simple eigenvalue gives differentiable branches $\lambda(\varepsilon)$, $x(\varepsilon)$ with $\lambda(0) = \lambda$, $x(0) = x$.

**Step 1 — differentiate the eigen-relation.**

$$
(A + \varepsilon E)\, x(\varepsilon) = \lambda(\varepsilon)\, x(\varepsilon)
$$

Differentiate at $\varepsilon = 0$:

$$
E x + A \dot{x} = \dot{\lambda} x + \lambda \dot{x}
$$

**Step 2 — project onto the left eigenvector.** Multiply on the left by $y^{*}$ and use $y^{*}A = \lambda y^{*}$:

$$
y^{*}Ex + \lambda y^{*}\dot{x} = \dot{\lambda}\, y^{*}x + \lambda y^{*}\dot{x}
$$

The $\dot{x}$ terms cancel exactly — this is the whole trick — leaving

$$
\dot{\lambda} = \frac{y^{*}Ex}{y^{*}x}
$$

**Step 3 — take the worst case over $\Vert E \Vert_2 = 1$.** Since $\vert y^{*}Ex \vert \le \Vert y \Vert_2 \Vert E \Vert_2 \Vert x \Vert_2 = 1$ with equality for $E = yx^{*}$,

$$
\hat{\kappa}_{\lambda} = \sup_{\Vert E \Vert_2 = 1} \vert \dot{\lambda} \vert = \frac{1}{\vert y^{*}x \vert} \qquad \blacksquare
$$

**Interpretation.** $\vert y^{*}x \vert = \cos\vartheta$ where $\vartheta$ is the angle between the left and right eigenvectors. For a symmetric (more generally normal) matrix, $y = x$, $\cos\vartheta = 1$, and $\hat{\kappa}_{\lambda} = 1$: symmetric eigenvalue problems are perfectly conditioned in the absolute sense, so `numpy.linalg.eigh` is trustworthy to $O(\Vert A \Vert u)$ regardless of $\kappa(A)$. For a strongly non-normal matrix the left and right eigenvectors become nearly orthogonal, $\cos\vartheta \to 0$, and eigenvalues become arbitrarily sensitive — the regime where *pseudospectra*, not eigenvalues, describe the observable behaviour (transient growth in fluid stability, non-normal recurrent dynamics).

**A worked pair.**

$$
A = \begin{pmatrix} 1 & 10^{8} \\ 0 & 1 + 10^{-8} \end{pmatrix}
$$

has eigenvalues $1$ and $1 + 10^{-8}$; the right eigenvectors are nearly parallel and $\vert y^{*}x \vert \approx 10^{-16}$, so $\hat{\kappa}_{\lambda} \approx 10^{16}$: rounding the matrix entries alone can move these eigenvalues into a complex conjugate pair. Its symmetric counterpart $\mathrm{diag}(1, 1 + 10^{-8})$ has $\hat{\kappa}_{\lambda} = 1$.

## 4. Computational & Algorithmic Insights

### 4.1 Estimating $\kappa$ without paying for it

- `numpy.linalg.cond(A)` computes the **full SVD**: $O(n^{3})$ with a large constant. Fine for diagnostics on small matrices, wasteful inside a solver.
- LAPACK's `*gecon` estimates $\Vert A^{-1} \Vert_1$ from an existing LU factorization in $O(n^{2})$ using Hager–Higham's 1-norm power iteration; SciPy exposes it indirectly and warns `LinAlgWarning: Ill-conditioned matrix` when $1/\kappa$ falls below machine precision. Cost: a few extra triangular solves.
- `numpy.linalg.matrix_rank` and `lstsq(..., rcond=r)` use the singular-value cutoff $\sigma_i \gt r\,\sigma_{\max}$ — implicitly a decision about the largest $\kappa$ you are willing to trust. The default `rcond=None` uses $\max(m,n)\,\varepsilon$.
- Cheap proxies: for symmetric positive definite $A$, the ratio of the largest to smallest diagonal entry is a lower bound; a few Lanczos/power iterations estimate $\sigma_{\max}$ and $\sigma_{\min}$ to one digit at $O(n^{2})$ cost, which is all the digit rule needs.

> Rule: report $\log_{10}\kappa$, not $\kappa$. A one-digit estimate of the exponent is all the accuracy the digit rule can use.

### 4.2 Practical conditioning repairs, in order of preference

1. **Rescale (equilibrate).** Row/column scaling changes $\kappa$ dramatically: a system in mixed units (metres and nanometres) can have $\kappa = 10^{9}$ purely from units. Van der Sluis's theorem: scaling rows to unit norm brings $\kappa_\infty$ within a factor $n$ of the optimum over all row scalings. **Standardize features before regression** — the same theorem in ML clothing.
2. **Change basis.** Monomials $\to$ Chebyshev/Legendre for polynomial fitting; raw correlated features $\to$ PCA/whitened features; naive parameterizations $\to$ orthogonal ones.
3. **Choose the factorization that does not square $\kappa$.** `lstsq` (QR/SVD) over normal equations; `solve` over `inv`; Cholesky only when the Gram matrix is genuinely the object of interest and is well conditioned.
4. **Precondition.** Solve $M^{-1}Ax = M^{-1}b$ with $M \approx A$ cheap to invert (Jacobi, incomplete Cholesky, multigrid). $\kappa(M^{-1}A) \ll \kappa(A)$ drives the convergence rate of CG, whose iteration count scales as $\sqrt{\kappa}$.
5. **Regularize.** Ridge/Tikhonov, truncated SVD, early stopping — accept bias in exchange for a bounded $\kappa$ (Derivation 3.6).
6. **Iterative refinement.** Compute $\hat{x}$, form the residual $r = b - A\hat{x}$ *in higher precision*, solve $A d = r$ with the existing factorization, update $\hat{x} \mathrel{+}= d$. Each pass gains roughly $\log_{10}(1/(\kappa u))$ digits until the limit set by the residual precision. This is the classical route — and the modern GPU route, where an fp16 or tf32 factorization is refined to fp64 accuracy.
7. **Raise precision.** Last resort on CPUs (`float128`, `mpmath`); costly, but the only option when the problem is intrinsically ill-conditioned and the answer is genuinely needed.

### 4.3 Conditioning of common numerical tasks

| Task | Condition number | Notes |
|---|---|---|
| Solve $Ax = b$ | $\kappa(A)$ | attained; use `solve`, never `inv` |
| Matrix–vector product $Ax$ | $\le \kappa(A)$, $= \frac{\Vert A \Vert \Vert x \Vert}{\Vert Ax \Vert}$ | small when $x$ aligns with a dominant singular direction |
| Least squares, small residual | $\approx \kappa_2(X)$ | QR achieves it |
| Least squares, large residual | up to $\kappa_2(X)^{2}$ | intrinsic, not a method artifact |
| Normal equations solve | $\kappa_2(X)^{2}$ | always avoidable |
| Matrix inversion | $\kappa(A)$ per entry | plus extra $O(n)$ growth in practice |
| Symmetric eigenvalues | $1$ (absolute) | `eigh` is excellent |
| Non-normal eigenvalues | $1/\vert y^{*}x \vert$ | can be astronomically large |
| Eigenvector $x_i$ | $1/\mathrm{gap}_i$ | clusters destroy individual vectors, not subspaces |
| Matrix exponential $e^{A}$ | up to $\Vert A \Vert e^{\Vert A \Vert}$ | "nineteen dubious ways" — scaling and squaring |
| Polynomial roots from coefficients | $\vert a_k r^{k-1}/p'(r) \vert$ | infinite at multiple roots |
| Summation $\sum x_i$ | $\frac{\sum \vert x_i \vert}{\vert \sum x_i \vert}$ | Topic 02's $\kappa_{\mathrm{sum}}$ |

## 5. Real-World Physics & AI/ML Applications

### 5.1 Collinear features and the geometry of regression

Two nearly duplicated feature columns $x_2 = x_1 + \epsilon z$ give $\sigma_{\min}(X) \approx \epsilon \Vert z \Vert$ while $\sigma_{\max}$ is $O(\Vert X \Vert)$, so $\kappa_2(X) \approx \Vert X \Vert/(\epsilon \Vert z \Vert)$. Fitted coefficients then satisfy $\beta_1 + \beta_2 \approx \text{const}$ but their individual values swing wildly — the textbook "unstable coefficients under multicollinearity" is *precisely* the perturbation bound of Theorem 2.4, with statistical noise playing the role of $\Delta b$.

Three standard responses map exactly onto Section 4.2: standardize columns (rescale), drop or merge duplicated features (change basis / reduce rank), or add $\lambda \Vert \beta \Vert^{2}$ (regularize). Note that *predictions* $X\hat{\beta}$ can remain well determined while *coefficients* are meaningless: the forward map is well conditioned in the directions the data actually explores. This is the difference between the conditioning of $\beta \mapsto X\beta$ and of $y \mapsto \beta$.

### 5.2 Hessian conditioning and the speed of gradient descent

For a quadratic $f(w) = \tfrac{1}{2} w^{\top} H w$ with $H \succ 0$, gradient descent with the optimal fixed step contracts the error by

$$
\frac{\Vert w_{k+1} - w^{*} \Vert}{\Vert w_k - w^{*} \Vert} \le \frac{\kappa - 1}{\kappa + 1}, \qquad \kappa = \kappa_2(H) = \frac{\lambda_{\max}}{\lambda_{\min}}
$$

so reaching accuracy $\epsilon$ takes $O(\kappa \log(1/\epsilon))$ iterations; heavy-ball/Nesterov momentum improves this to $O(\sqrt{\kappa}\log(1/\epsilon))$, and conjugate gradient likewise scales as $\sqrt{\kappa}$. Measured Hessians of trained networks routinely show $\lambda_{\max}/\lambda_{\text{bulk}} \gtrsim 10^{4}$–$10^{6}$ with a near-null bulk, which is why:

- **Normalization layers** (BatchNorm, LayerNorm) work partly by whitening activations, compressing the spectrum of the effective curvature.
- **Adam / RMSProp** apply a diagonal preconditioner $\mathrm{diag}(1/\sqrt{v_t + \epsilon})$ — a cheap approximation of $H^{-1/2}$, i.e. exactly step 4 of Section 4.2.
- **K-FAC, Shampoo, and second-order methods** build better preconditioners at higher cost.
- **Learning-rate warmup** avoids the early phase where $\lambda_{\max}$ estimates are unreliable and a too-large step diverges along the sharpest direction.

Here $\kappa$ controls *convergence speed*, not floating-point accuracy — the same quantity, a different consequence, and the reason "ill-conditioned" is a complaint in both numerical analysis and optimization.

### 5.3 Physics: inverse problems, deconvolution and Tikhonov's origin

Forward physics is usually smoothing: heat diffuses, optics blurs, seismic waves attenuate. Smoothing operators have singular values decaying to zero (often exponentially for the backward heat equation), so the **inverse** problem — recover the source from the observation — is ill-conditioned or genuinely ill-posed in Hadamard's sense. A CT reconstruction, a deblurring, or a spectral unmixing amplifies measurement noise by $1/\sigma_{\min}$.

Tikhonov introduced $\lambda \Vert \beta \Vert^{2}$ in exactly this context in 1963, decades before it entered statistics as ridge regression; the SVD filter factors $\sigma_i^{2}/(\sigma_i^{2} + \lambda)$ of Derivation 3.6 are the same Wiener-filter weights used in image restoration. The **discrepancy principle** — choose $\lambda$ so that $\Vert X\beta_\lambda - y \Vert \approx$ the known noise level — is the physics community's version of cross-validation, and the **L-curve** is its diagnostic plot. Modern learned reconstruction (plug-and-play priors, diffusion-model posterior sampling) replaces the quadratic penalty by a learned prior but plays exactly the same conditioning role: restrict the solution to a subspace where the forward operator is invertible with bounded amplification.

### 5.4 Covariance matrices, Gaussian processes, and the "jitter" hack

A Gaussian-process regressor must factor the kernel matrix $K + \sigma_n^{2}I$. With a smooth kernel (RBF) and nearby input points, $K$ has exponentially decaying eigenvalues: $\kappa_2(K)$ can exceed $10^{16}$ for a few hundred points, and Cholesky fails with "matrix is not positive definite" — not because the mathematics is wrong (an RBF kernel matrix is positive definite for distinct points) but because rounding pushed the smallest eigenvalue below zero.

The universal fix is to add **jitter** $\varepsilon I$ with $\varepsilon \sim 10^{-6}\,\mathrm{tr}(K)/n$, which is Derivation 3.6 with $\lambda = \varepsilon$: it caps the condition number at $\approx \lambda_{\max}/\varepsilon$ and is statistically interpretable as a small observation-noise floor. The same pattern recurs across ML:

- **Whitening / PCA**: dropping components with $\sigma_i \lt \tau$ is truncated SVD.
- **Natural gradient / K-FAC**: damping $F + \lambda I$ on the Fisher matrix.
- **Adam's $\epsilon$**: $\hat{m}/(\sqrt{\hat{v}} + \epsilon)$ bounds the preconditioner's condition number by $\sqrt{v_{\max}}/\epsilon$ (Topic 05 analyzes where the $\epsilon$ must sit).
- **Attention logits**: dividing by $\sqrt{d_k}$ keeps the softmax argument in a range where the Jacobian is well conditioned.

In every case a hyperparameter that looks statistical is doing double duty as a condition-number cap.

## 6. Canonical Literature Mapping & References

| Concept in this notebook | Canonical source | Location |
|---|---|---|
| Relative condition number, $\kappa_f = \Vert J \Vert \Vert x \Vert / \Vert f(x) \Vert$ | Trefethen & Bau, *Numerical Linear Algebra* (1997) | Lecture 12 |
| $\kappa(A) = \Vert A \Vert \Vert A^{-1} \Vert$, $\kappa_2 = \sigma_{\max}/\sigma_{\min}$ | Trefethen & Bau (1997) | Lecture 12; Golub & Van Loan, Sec. 2.6 |
| Linear-system perturbation theorem | Higham, *Accuracy and Stability of Numerical Algorithms* (2002) | Ch. 7, Thm. 7.2 |
| Distance to singularity | Kahan (1966); Golub & Van Loan (2013) | Sec. 2.6.2 |
| Backward stability, forward $\lesssim \kappa \times$ backward | Trefethen & Bau (1997) | Lectures 14–15 |
| Least-squares perturbation, $\kappa^{2}\tan\theta$ term | Golub & Van Loan (2013); Higham (2002) | Sec. 5.3; Ch. 20 |
| Normal equations squaring $\kappa$ | Trefethen & Bau (1997) | Lecture 19 |
| Eigenvalue condition number $1/\vert y^{*}x \vert$, Bauer–Fike | Golub & Van Loan (2013); Demmel (1997) | Sec. 7.2; Ch. 4 |
| Pseudospectra for non-normal operators | Trefethen & Embree, *Spectra and Pseudospectra* (2005) | Part I |
| Wilkinson's polynomial, root conditioning | Wilkinson, *Rounding Errors in Algebraic Processes* (1963) | Ch. 2 |
| Tikhonov regularization, filter factors | Hansen, *Rank-Deficient and Discrete Ill-Posed Problems* (1998) | Ch. 4–5 |
| Scaling/equilibration bounds | van der Sluis (1969); Higham (2002) | Ch. 7.3 |
| Condition estimation in $O(n^{2})$ | Hager (1984); Higham & Tisseur (2000) | LAPACK `*gecon` |
| $\kappa$ and gradient-descent rate | Nocedal & Wright, *Numerical Optimization* (2006) | Sec. 3.3 |

**Continue to** [Topic 04: Vectorization and NumPy Performance](../04_vectorization_and_numpy_performance/README.md) — from *how accurate* to *how fast*, and why the two are decided by the same memory hierarchy.